In [ ]:
import os
# where the raw pair tensors live; see pipeline/embeddings/README.md
EMB_ROOT = os.environ['PREPIBIND_EMB_ROOT']   # the embedding stores; no default, see pipeline/embeddings/README.md
import h5py
import numpy as np
from glob import glob
import os
from tqdm import tqdm

def pair2side(input_files, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    for input_file in input_files:
        output_file = os.path.join(output_folder, os.path.splitext(os.path.basename(input_file))[0].replace('pair', 'pair_side') + '.h5')
        with h5py.File(input_file, 'r') as f:
            # use tqdm to show progress bar
            with tqdm(total=len(f.keys()), desc=f'Processing {os.path.basename(input_file)}') as pbar:
                for key in f.keys():
                    data = f[key][...]
                    side_data_1 = np.mean(data, axis=0)
                    side_data_2 = np.mean(data, axis=1)
                    side_data = np.concatenate((side_data_1, side_data_2), axis=-1)
                    pbar.update(1)
                    # save side_data to new file
                    with h5py.File(output_file, 'a') as new_f:
                        new_f.create_dataset(key, data=side_data)

input_files = glob(os.path.join(EMB_ROOT, 'pair', '*pair*.h5'))
input_files = sorted(input_files)
output_folder = os.path.join(EMB_ROOT, 'side')
pair2side(input_files, output_folder)